In [1]:
!pip install fastf1 scikit-learn pandas joblib pyarrow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.0/136.0 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00


In [2]:
"""
Train a PRE-RACE finishing-position predictor for F1 races.

This is separate from your existing overtake model. It only uses data that
is available BEFORE a race starts: grid position, recent driver/team form,
circuit history, and season standings — never lap times or telemetry from
the race itself (that would be data leakage, since you don't have it for
a race that hasn't happened yet).

RUN THIS LOCALLY OR IN COLAB — it needs internet access to download data
via FastF1, and downloading several seasons of results can take a while
(but it's just results, not lap-by-lap/telemetry, so it's much faster
than what app.py currently loads).

Outputs (copy both into your repo root, next to app.py):
  - f1_race_predictor.pkl       the trained model
  - historical_features.parquet rolling per-driver/team stats used to
                                 build features for races that haven't
                                 happened yet
"""

import os
import fastf1
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import joblib

os.makedirs("cache", exist_ok=True)
fastf1.Cache.enable_cache("cache")

# Adjust as you like. More years = more training data, but older seasons
# are less representative of current cars/regulations.
YEARS = range(2019, 2026)

FEATURES = [
    "GridPosition",
    "DriverAvgFinishLast5",
    "DriverAvgGridLast5",
    "DriverDNFRateLast10",
    "DriverPointsCumSeason",
    "TeamAvgFinishLast5",
    "TeamPointsCumSeason",
    "CircuitAvgFinish",
    "DriverRaceCount",
]


def collect_race_results(years):
    """Pull just the results table (grid, finish, points, status) for
    every race in the given years — no laps/telemetry needed."""
    rows = []
    for year in years:
        schedule = fastf1.get_event_schedule(year, include_testing=False)
        schedule = schedule[schedule["EventFormat"] != "testing"]
        for _, event in schedule.iterrows():
            rnd = int(event["RoundNumber"])
            if rnd == 0:
                continue
            try:
                session = fastf1.get_session(year, rnd, "R")
                session.load(laps=False, telemetry=False, weather=False, messages=False)
                results = session.results.copy()
                if results is None or results.empty:
                    continue
                results["Year"] = year
                results["Round"] = rnd
                results["EventName"] = event["EventName"]
                rows.append(results)
                print(f"  loaded {year} R{rnd} {event['EventName']}")
            except Exception as e:
                print(f"  skipping {year} R{rnd} ({event['EventName']}): {e}")
    return pd.concat(rows, ignore_index=True)


def build_features(all_results):
    """Engineer rolling, leakage-free pre-race features.

    Every rolling/cumulative stat is computed with .shift() first, so a
    race's features only ever reflect what was KNOWN before that race.
    """
    df = all_results.copy()
    df["Position"] = pd.to_numeric(df["Position"], errors="coerce")
    df["GridPosition"] = pd.to_numeric(df["GridPosition"], errors="coerce")
    df["Points"] = pd.to_numeric(df["Points"], errors="coerce").fillna(0)
    df["DNF"] = df["Status"].apply(
        lambda s: 0 if ("finish" in str(s).lower() or "lap" in str(s).lower()) else 1
    )
    df = df.sort_values(["Year", "Round"]).reset_index(drop=True)

    df["DriverAvgFinishLast5"] = (
        df.groupby("Abbreviation")["Position"]
        .transform(lambda s: s.shift().rolling(5, min_periods=1).mean())
    )
    df["DriverAvgGridLast5"] = (
        df.groupby("Abbreviation")["GridPosition"]
        .transform(lambda s: s.shift().rolling(5, min_periods=1).mean())
    )
    df["DriverDNFRateLast10"] = (
        df.groupby("Abbreviation")["DNF"]
        .transform(lambda s: s.shift().rolling(10, min_periods=1).mean())
    )
    df["DriverPointsCumSeason"] = (
        df.groupby(["Year", "Abbreviation"])["Points"]
        .transform(lambda s: s.shift().cumsum().fillna(0))
    )
    df["TeamAvgFinishLast5"] = (
        df.groupby("TeamName")["Position"]
        .transform(lambda s: s.shift().rolling(5, min_periods=1).mean())
    )
    df["TeamPointsCumSeason"] = (
        df.groupby(["Year", "TeamName"])["Points"]
        .transform(lambda s: s.shift().cumsum().fillna(0))
    )
    df["CircuitAvgFinish"] = (
        df.groupby(["Abbreviation", "EventName"])["Position"]
        .transform(lambda s: s.shift().expanding().mean())
    )
    df["DriverRaceCount"] = df.groupby("Abbreviation").cumcount()

    return df


def main():
    print("Collecting historical race results (this can take a while)...")
    raw = collect_race_results(YEARS)
    df = build_features(raw)

    model_df = df.dropna(subset=["Position"]).copy()
    for c in FEATURES:
        model_df[c] = model_df[c].fillna(model_df[c].median())

    X = model_df[FEATURES]
    y = model_df["Position"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.15, random_state=42
    )

    model = GradientBoostingRegressor(
        n_estimators=300, max_depth=3, learning_rate=0.05, random_state=42
    )
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    print(f"\nValidation MAE (finishing position): {mae:.2f} positions")

    joblib.dump({"model": model, "features": FEATURES}, "f1_race_predictor.pkl")
    print("Saved f1_race_predictor.pkl")

    # The app needs the latest known per-driver/team rolling stats to build
    # a feature row for an upcoming race that hasn't happened yet.
    df.to_parquet("historical_features.parquet", index=False)
    print("Saved historical_features.parquet")


if __name__ == "__main__":
    main()

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for Australian Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['77', '44', '33', '5', '16', '20', '27', '7', '18', '26', '10', '4', '11', '23', 

  loaded 2019 R1 Australian Grand Prix


req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['44', '77', '16', '33', '5', '4', '7', '10', '23', '11', '99', '26', '20', '18', '63', '88', '27', '3', '55', '8']
INFO:fastf1.fastf1.core:Finished loading data for 20 drivers: ['44', '77', '16', '33', '5', '4', '7', '10', '23', '11', '99', '26', '20', '18', '63', '88', '27', '3', '55', '8']
core           INFO 	Loading data for Chinese Grand Prix - Race [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for Chinese Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for

  loaded 2019 R2 Bahrain Grand Prix


req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['44', '77', '5', '33', '16', '10', '3', '11', '7', '23', '8', '18', '20', '55', '99', '63', '88', '4', '26', '27']
INFO:fastf1.fastf1.core:Finished loading data for 20 drivers: ['44', '77', '5', '33', '16', '10', '3', '11', '7', '23', '8', '18', '20', '55', '99', '63', '88', '4', '26', '27']
core           INFO 	Loading data for Azerbaijan Grand Prix - Race [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for Azerbaijan Grand Prix - Race [v3.8.3]
req            INFO 	No cached data fou

...

  loaded 2025 R21 São Paulo Grand Prix


req            INFO 	Data has been written to cache!
INFO:fastf1.fastf1.req:Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
INFO:fastf1.api:Fetching driver list...
core        WARNING 	Failed to load extended driver information!
DEBUG:fastf1.fastf1.core:Exception while loading driver list
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/fastf1/core.py", line 2459, in _drivers_from_f1_api
    f1di = api.driver_info(self.api_path, livedata=livedata)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/req.py", line 459, in _cached_api_request
    data = func(api_path, **func_kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastf1/_api.py", line 1562, in driver_info
  

  skipping 2025 R22 (Las Vegas Grand Prix): any API: 500 calls/h


core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
INFO:fastf1.fastf1.core:Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
INFO:fastf1.fastf1.req:No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
INFO:fastf1.api:Fetching session info data...


  skipping 2025 R23 (Qatar Grand Prix): any API: 500 calls/h
  skipping 2025 R24 (Abu Dhabi Grand Prix): any API: 500 calls/h

Validation MAE (finishing position): 3.29 positions
Saved f1_race_predictor.pkl
Saved historical_features.parquet
